In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_excel("WineData.xlsx")
df.head()

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,red,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,red,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,red,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,red,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,red,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [4]:
df['type'] = df['type'].astype('category').cat.codes
# Converts wine types "red" and white" into numbers 0 and 1

df.head()

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,0,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,0,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,0,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [6]:
df['target_quality'] = pd.cut( # Create new column 'target_quality'
    df['quality'], 
    bins=[2,4,6,9], # Translates new number as an integer between ranges 
    labels=[0,1,2],
).astype(int)

In [7]:
df.head()

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,target_quality
0,0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1
1,0,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,0,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,1
3,0,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1
4,0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1


In [10]:
X = df.drop(["quality","target_quality"],axis=1).values # Making x column where it lists all columns except those 2?
y = df["target_quality"].values # Making y column

In [11]:
X = (X-X.mean(axis=0))/X.std() # Standardization formula 

In [14]:
y_onehot = np.zeros((y.size,3))
y_onehot[np.arange(y.size), y]=1
# Classes 0,1,2 
# For a data point, Class 1 is found as [0,1,0] and Class 2 is found as [0,0,1]

In [15]:
y_onehot

array([[0., 1., 0.],
       [0., 1., 0.],
       [0., 1., 0.],
       ...,
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.]])

In [16]:
np.random.seed(11242025)
indices = np.random.permutation(len(X))
train_size = int(0.8*len(X))

In [20]:
train_idx, test_idx = indices[:train_size], indices[train_size:] 
# Set variables 'train_idx' 'test_idx" with indices before last column as X and last index as y

X_train, y_train = X[train_idx], y_onehot[train_idx]

X_test, y_test = X[test_idx], y_onehot[train_size:]

In [40]:
# Activation functions

# Relu function
def relu(z1): 
    return np.maximum(0,z1)

# Relu derivative function
def relu_derivative(z2): 
    return (z2>0).astype(float)

# Softmax function
def softmax(z3): # (sigmiod  function max)
    exp=np.exp(z3-np.max(z3,axis=1,keepdims=True))
    return exp/np.sum(exp,axis=1,keepdims=True)

In [41]:
# Building ANN (2 Layer)
input_dim = X_train.shape[1]
hidden_dim = 16
output_dim = 3

In [42]:
# Initial weights 

# Layer 1 - Input layer
W1 = np.random.randn(input_dim,hidden_dim)*0.1
b1 = np.zeros((1,hidden_dim,))

# Layer 2 - Output layer
W2 = np.random.randn(hidden_dim,output_dim)*0.1
b2 = np.zeros((1,output_dim,))

lr = 0.005 # Learning rate
epochs = 3000 # Num of times the ANN will be trained on the data

In [43]:
# Train loops
for epoch in range(epochs): 
    z1 = X_train.dot(W1) + b1
    a1 = relu(z1)

    z2 = a1.dot(W2) + b2
    y_pred = softmax(z2)

    # Compute Loss (entropy) 
    loss = -np.mean(np.sum(y_train*np.log(y_pred + 1e-8),axis=1))

    # Backpropagation 
    dz2 = y_pred - y_train
    dW2 = a1.T.dot(dz2)/len(X_train)
    db2 = np.mean(dz2,axis=0,keepdims=True)

    dz1 = dz2.dot(W2.T)*relu_derivative(z1)
    dW1 = X_train.T.dot(dz1)/len(X_train)
    db1 = np.mean(dz1,axis=0,keepdims=True)

    # Gradient desent update
    # Layer 2
    W2 -= lr*dW2
    b2 -= lr*db2

    # Layer 1
    W1 -= lr*dW1
    b1 -= lr*db1

    # Other stuff...
    if epoch % 500 == 0:
        print(f"Epoch{epoch},Loss{loss:.4f}")

Epoch0,Loss1.0892
Epoch500,Loss0.7231
Epoch1000,Loss0.6698
Epoch1500,Loss0.6553
Epoch2000,Loss0.6487
Epoch2500,Loss0.6452
